In [ ]:
# Lecture 23 — Caching & Persistence: same job with and without cache
# Spark 101 · Module 7 · Lecture 23
#
# Import this file into a Databricks workspace as a notebook.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.
# In VS Code / PyCharm this stays a plain Python file — same source, two homes.
#
# BIG-DATA DEMO: use data big enough that the cache delta is obvious.
# Warn about DBU cost on screen and keep auto-termination on.

In [ ]:
print("Spark version:", spark.version)
# Databricks gives you `spark` for free — no SparkSession.builder needed.

In [ ]:
# --- An expensive chain: a groupBy (shuffle) over many rows ----------------
import time
from pyspark.sql.functions import col, count, sum as _sum

df = spark.range(0, 20_000_000).withColumn("key", (col("id") % 500))
expensive = (
    df.filter(col("id") % 2 == 0)                 # narrow
      .groupBy("key")                              # WIDE — shuffle
      .agg(count("*").alias("n"), _sum("id").alias("total"))
)

In [ ]:
# --- WITHOUT cache: three actions -> three full recomputations --------------
t0 = time.time()
expensive.count()                      # runs the whole chain
expensive.orderBy("total").take(5)     # runs the whole chain AGAIN
expensive.agg(_sum("n")).collect()     # ...and a THIRD time
print(f"no cache, 3 actions: {time.time() - t0:.1f}s")

In [ ]:
# --- WITH cache: compute once, reuse ---------------------------------------
cached = expensive.cache()   # LAZY — marks it; nothing stored yet

t0 = time.time()
cached.count()                         # computes AND stores the result
cached.orderBy("total").take(5)        # reads from cache — skips the chain
cached.agg(_sum("n")).collect()        # reads from cache again
print(f"cached, 3 actions:  {time.time() - t0:.1f}s")
# First action is the same speed; the REPEATS are what get fast.

In [ ]:
# --- Release it when done — memory is finite -------------------------------
cached.unpersist()
print("cache released")

In [ ]:
# --- persist() with a storage level — safe default for big data ------------
from pyspark.storagelevel import StorageLevel

safe = expensive.persist(StorageLevel.MEMORY_AND_DISK)  # spill instead of drop
safe.count()   # materialize
print("persisted MEMORY_AND_DISK; storage level:", safe.storageLevel)
safe.unpersist()

In [ ]:
# Check whether a DataFrame is cached.
c2 = expensive.cache(); c2.count()
print("is_cached after cache:", c2.is_cached)
c2.unpersist(); print("is_cached after unpersist:", c2.is_cached)

In [ ]:
# Cache a SQL temp view with CACHE TABLE.
expensive.createOrReplaceTempView("agg_v")
spark.sql("CACHE TABLE agg_v")
spark.sql("SELECT count(*) AS n FROM agg_v").show()
spark.sql("UNCACHE TABLE agg_v")

In [ ]:
# A different storage level: DISK_ONLY (survives memory pressure, slower reads).
from pyspark.storagelevel import StorageLevel
d = expensive.persist(StorageLevel.DISK_ONLY)
d.count()
print("storage level:", d.storageLevel)
d.unpersist()

In [ ]:
# Recap:
#   - Lazy means each action recomputes the full lineage from source.
#   - cache()/persist() store the result so repeats skip the recompute.
#   - cache() is lazy — the first action stores it; later actions read it.
#   - Cache what you REUSE; unpersist() when done. Don't cache single-use data.
# Next lecture: broadcast joins — skip the shuffle on big-vs-small joins.